# 02 Data Cleaning

This notebook builds the machine-learning dataset in stages. It starts by loading and merging the raw files, then cleans the merged item-level table, and then creates first-pass engineered features.

## 1. Load Raw CSV Files

Loading each raw table separately keeps the original source data visible before any joins happen. The product category translation table is also loaded so category names can be translated during the merge stage.

In [28]:
from pathlib import Path  # Import Path to build reliable file paths across operating systems.

import pandas as pd  # Import pandas to load CSV files and merge tabular data.

raw_data_dir = Path("../data/raw")  # Store the relative path to the raw data folder.

orders = pd.read_csv(raw_data_dir / "olist_orders_dataset.csv")  # Load the raw orders table.
order_items = pd.read_csv(raw_data_dir / "olist_order_items_dataset.csv")  # Load the raw order items table.
products = pd.read_csv(raw_data_dir / "olist_products_dataset.csv")  # Load the raw products table.
sellers = pd.read_csv(raw_data_dir / "olist_sellers_dataset.csv")  # Load the raw sellers table.
customers = pd.read_csv(raw_data_dir / "olist_customers_dataset.csv")  # Load the raw customers table.
category_translation = pd.read_csv(raw_data_dir / "product_category_name_translation.csv")  # Load product category translations.

## 2. Run Integrity Checks Before Merging

Integrity checks help confirm whether the join keys behave as expected before tables are combined. Primary key checks show whether IDs are unique in their source tables, while orphan checks show whether the order items table contains references that cannot be matched to orders, products, or sellers.

In [29]:
primary_key_checks = {  # Create a dictionary describing the expected primary key in each source table.
    "orders.order_id": orders["order_id"].is_unique,  # Check whether each order_id appears once in the orders table.
    "customers.customer_id": customers["customer_id"].is_unique,  # Check whether each customer_id appears once in the customers table.
    "products.product_id": products["product_id"].is_unique,  # Check whether each product_id appears once in the products table.
    "sellers.seller_id": sellers["seller_id"].is_unique,  # Check whether each seller_id appears once in the sellers table.
}  # Close the dictionary of primary key checks.

print("Primary key uniqueness checks:")  # Print a heading for the primary key check results.
for key_name, is_unique in primary_key_checks.items():  # Loop through each primary key check result.
    print(f"{key_name} unique: {is_unique}")  # Print whether the key is unique in its own source table.

orphan_order_id_count = (~order_items["order_id"].isin(orders["order_id"])).sum()  # Count order item rows with order_id values missing from orders.
orphan_product_id_count = (~order_items["product_id"].isin(products["product_id"])).sum()  # Count order item rows with product_id values missing from products.
orphan_seller_id_count = (~order_items["seller_id"].isin(sellers["seller_id"])).sum()  # Count order item rows with seller_id values missing from sellers.

print("\nOrphan row checks in order_items:")  # Print a heading for orphan row check results.
print(f"Rows with order_id not found in orders: {orphan_order_id_count}")  # Print the count of order item rows with unmatched order_id values.
print(f"Rows with product_id not found in products: {orphan_product_id_count}")  # Print the count of order item rows with unmatched product_id values.
print(f"Rows with seller_id not found in sellers: {orphan_seller_id_count}")  # Print the count of order item rows with unmatched seller_id values.

Primary key uniqueness checks:
orders.order_id unique: True
customers.customer_id unique: True
products.product_id unique: True
sellers.seller_id unique: True

Orphan row checks in order_items:
Rows with order_id not found in orders: 0
Rows with product_id not found in products: 0
Rows with seller_id not found in sellers: 0


## 3. Merge Raw Tables Into One Item-Level Table

The merge starts from orders and keeps only records that successfully match customers, order items, products, and sellers through inner joins. The category translation is added with a left join so rows are kept even if a category translation is missing.

In [30]:
merged_item_level = orders.merge(customers, on="customer_id", how="inner")  # Inner-join orders to customers using customer_id.
merged_item_level = merged_item_level.merge(order_items, on="order_id", how="inner")  # Inner-join the current table to order items using order_id.
merged_item_level = merged_item_level.merge(products, on="product_id", how="inner")  # Inner-join the current table to products using product_id.
merged_item_level = merged_item_level.merge(sellers, on="seller_id", how="inner")  # Inner-join the current table to sellers using seller_id.
merged_item_level = merged_item_level.merge(category_translation, on="product_category_name", how="left")  # Left-join product category translations using product_category_name.

## 4. Print the Merged Table Shape

Printing the merged shape confirms how many item-level rows and columns exist after the requested joins. This gives a baseline for later cleaning, feature engineering, and order-level aggregation.

In [31]:
print(f"Merged item-level table shape: {merged_item_level.shape}")  # Print the number of rows and columns in the merged item-level table.

Merged item-level table shape: (112650, 30)


## Cleaning Stage

This stage starts cleaning the merged item-level table so later feature engineering and order-level aggregation can use valid delivered orders.

## 1. Drop Orders Without Actual Delivery Dates

Rows with missing `order_delivered_customer_date` represent orders that were not delivered, so they cannot be used to calculate actual delivery time for this prediction task.

In [32]:
cleaned_item_level = merged_item_level.dropna(subset=["order_delivered_customer_date"]).copy()  # Keep only rows where the order has an actual customer delivery timestamp.

## 2. Convert Date Columns to Datetime

The timestamp columns must be real datetime values before pandas can calculate time differences correctly.

In [33]:
cleaned_item_level["order_purchase_timestamp"] = pd.to_datetime(cleaned_item_level["order_purchase_timestamp"])  # Convert purchase timestamps from strings to datetime values.
cleaned_item_level["order_delivered_customer_date"] = pd.to_datetime(cleaned_item_level["order_delivered_customer_date"])  # Convert delivered timestamps from strings to datetime values.
cleaned_item_level["order_estimated_delivery_date"] = pd.to_datetime(cleaned_item_level["order_estimated_delivery_date"])  # Convert estimated delivery dates from strings to datetime values.

## 3. Create the Delivery Time Target

`delivery_days` is the machine learning target variable because it measures the actual number of days between purchase and customer delivery.

In [34]:
delivery_time_delta = cleaned_item_level["order_delivered_customer_date"] - cleaned_item_level["order_purchase_timestamp"]  # Calculate the time difference between delivery and purchase.
cleaned_item_level["delivery_days"] = delivery_time_delta.dt.total_seconds() / 86400  # Convert the delivery time difference from seconds into days as a float.

## 4. Remove Impossible Delivery Durations

Zero or negative delivery durations are impossible for this target and should be treated as data errors before modeling.

In [35]:
cleaned_item_level = cleaned_item_level[cleaned_item_level["delivery_days"] > 0].copy()  # Keep only rows with positive delivery time values.

## 5. Drop Missing Product Dimensions and Create Product Volume

Product dimensions are needed to calculate product volume, so rows missing length, height, or width cannot support that engineered feature.

In [36]:
cleaned_item_level = cleaned_item_level.dropna(subset=["product_length_cm", "product_height_cm", "product_width_cm"]).copy()  # Drop rows missing any required product dimension.
cleaned_item_level["product_volume_cm3"] = cleaned_item_level["product_length_cm"] * cleaned_item_level["product_height_cm"] * cleaned_item_level["product_width_cm"]  # Calculate product volume in cubic centimeters.

## 6. Fill Missing English Product Categories

Missing translated categories should be labeled as `unknown` so the rows stay available while still marking the missing category information clearly.

In [37]:
cleaned_item_level["product_category_name_english"] = cleaned_item_level["product_category_name_english"].fillna("unknown")  # Replace missing English product category names with the label unknown.

## 7. Confirm Cleaning Results

The final checks confirm the cleaned item-level table shape and verify that the target variable has no missing or negative values.

In [38]:
delivery_days_missing_count = cleaned_item_level["delivery_days"].isna().sum()  # Count missing values in the delivery_days target column.
delivery_days_negative_count = (cleaned_item_level["delivery_days"] < 0).sum()  # Count negative values in the delivery_days target column.

print(f"Shape after cleaning: {cleaned_item_level.shape}")  # Print the number of rows and columns after the cleaning steps.
print(f"Missing delivery_days values: {delivery_days_missing_count}")  # Print the missing value count for the target variable.
print(f"Negative delivery_days values: {delivery_days_negative_count}")  # Print the negative value count for the target variable.

Shape after cleaning: (110178, 32)
Missing delivery_days values: 0
Negative delivery_days values: 0


## Feature Engineering Stage

This stage creates model-ready features from the cleaned item-level table while avoiding target leakage.

## 1. Create Order Day of Week

The purchase timestamp can capture weekly ordering patterns. Pandas stores Monday as 0 and Sunday as 6.

In [39]:
cleaned_item_level["order_day_of_week"] = cleaned_item_level["order_purchase_timestamp"].dt.dayofweek  # Extract the purchase day of week where Monday is 0 and Sunday is 6.

## 2. Create Weekend Order Flag

Weekend purchases may have different delivery behavior, so this feature marks only Saturday and Sunday as weekend orders. Friday is day 4, so it must stay coded as 0.

In [40]:
cleaned_item_level["is_weekend_order"] = cleaned_item_level["order_day_of_week"].isin([5, 6]).astype(int)  # Set weekend orders to 1 only for Saturday day 5 and Sunday day 6.

## 3. Create Same-State Delivery Flag

Deliveries where the seller and customer are in the same state may be faster than deliveries across state boundaries.

In [41]:
cleaned_item_level["same_state_delivery"] = (cleaned_item_level["seller_state"] == cleaned_item_level["customer_state"]).astype(int)  # Set same-state deliveries to 1 when seller_state equals customer_state.

## 4. Create Leave-One-Out Seller Average Delivery Days

A simple seller average would include the current row's own delivery time inside its own feature, which leaks part of the target into the model input. Leave-one-out avoids that by using the seller's other rows only. If a seller has only one row, there are no other seller rows to average, so the global mean delivery time is used as the fallback.

In [42]:
global_mean_delivery_days = cleaned_item_level["delivery_days"].mean()  # Calculate the overall average delivery time for one-row seller fallback values.
seller_delivery_total = cleaned_item_level.groupby("seller_id")["delivery_days"].transform("sum")  # Calculate each seller's total delivery_days and align it to every row.
seller_delivery_count = cleaned_item_level.groupby("seller_id")["delivery_days"].transform("count")  # Calculate each seller's row count and align it to every row.
seller_other_delivery_total = seller_delivery_total - cleaned_item_level["delivery_days"]  # Remove the current row's own delivery_days from the seller total.
seller_other_delivery_count = seller_delivery_count - 1  # Remove the current row from the seller count.
cleaned_item_level["seller_avg_delivery_days"] = seller_other_delivery_total / seller_other_delivery_count  # Calculate each seller's leave-one-out average delivery time.
cleaned_item_level["seller_avg_delivery_days"] = cleaned_item_level["seller_avg_delivery_days"].fillna(global_mean_delivery_days)  # Fill undefined one-row seller averages with the global mean.

## 5. Sanity Check Weekend Logic

This sanity check confirms that Friday orders are never marked as weekend orders.

In [43]:
friday_weekend_values = cleaned_item_level.loc[cleaned_item_level["order_day_of_week"] == 4, "is_weekend_order"].unique()  # Get unique weekend flag values for Friday rows only.
print(f"Unique is_weekend_order values for Friday orders: {friday_weekend_values}")  # Print the Friday sanity check values, which should only contain 0.

Unique is_weekend_order values for Friday orders: [0]


## Aggregation Stage

The cleaned table is currently item-level, so orders with multiple items appear multiple times. This stage aggregates the data to one row per `order_id` because `delivery_days` is an order-level target.

## 1. Define the Top Category Rule

The order-level `top_category` feature should use the most frequent English product category within each order. If a mode cannot be found, the category falls back to `unknown`.

In [44]:
def get_top_category(category_series):  # Define a helper function to find the most frequent category in one order.
    category_modes = category_series.mode(dropna=True)  # Calculate the mode after ignoring missing category values.
    if category_modes.empty:  # Check whether no mode was available for this order.
        return "unknown"  # Return unknown when the order has no usable category value.
    return category_modes.iloc[0]  # Return the first mode when one or more modes are available.

## 2. Aggregate to One Row per Order

Order-level fields use the first value because they are the same across every item in an order. Item-level fields are counted, summed, averaged, or summarized so each final row represents one order.

In [45]:
order_level_clean = cleaned_item_level.groupby("order_id").agg(  # Group item-level rows by order_id and aggregate them into one row per order.
    customer_id=("customer_id", "first"),  # Keep the first customer_id because it is the same for every item in an order.
    customer_state=("customer_state", "first"),  # Keep the first customer_state because it is the same for every item in an order.
    order_purchase_timestamp=("order_purchase_timestamp", "first"),  # Keep the first purchase timestamp because it is the same for every item in an order.
    order_day_of_week=("order_day_of_week", "first"),  # Keep the first order day of week because it is the same for every item in an order.
    is_weekend_order=("is_weekend_order", "first"),  # Keep the first weekend flag because it is the same for every item in an order.
    delivery_days=("delivery_days", "first"),  # Keep the first delivery_days value because delivery time belongs to the order.
    num_items=("order_item_id", "count"),  # Count item rows to get the number of items in each order.
    num_unique_sellers=("seller_id", "nunique"),  # Count distinct sellers in each order.
    num_unique_products=("product_id", "nunique"),  # Count distinct products in each order.
    total_price=("price", "sum"),  # Sum item prices to get total order price.
    avg_price=("price", "mean"),  # Average item prices within each order.
    total_freight_value=("freight_value", "sum"),  # Sum item freight values to get total order freight cost.
    avg_freight_value=("freight_value", "mean"),  # Average item freight values within each order.
    avg_product_weight_g=("product_weight_g", "mean"),  # Average product weights across the order's items.
    avg_product_volume_cm3=("product_volume_cm3", "mean"),  # Average product volumes across the order's items.
    pct_items_same_state=("same_state_delivery", "mean"),  # Average item same-state flags to get the share of same-state items.
    seller_avg_delivery_days=("seller_avg_delivery_days", "mean"),  # Average seller delivery history features across the order's items.
    top_category=("product_category_name_english", get_top_category),  # Use the most frequent English product category in the order.
).reset_index()  # Convert order_id from the grouped index back into a regular column.

order_level_clean["same_state_delivery"] = (order_level_clean["pct_items_same_state"] == 1.0).astype(int)  # Mark the whole order as same-state only when every item is same-state.

print(f"Final order-level table shape: {order_level_clean.shape}")  # Print the number of rows and columns in the final order-level table.

Final order-level table shape: (96460, 20)


## 3. Save Processed Tables

Save the cleaned item-level table for auditing and the aggregated order-level table for machine learning.

In [46]:
processed_data_dir = Path("../data/processed")  # Store the relative path to the processed data folder.
processed_data_dir.mkdir(parents=True, exist_ok=True)  # Create the processed data folder if it does not already exist.

item_level_output_path = processed_data_dir / "item_level_clean.csv"  # Define the output path for the cleaned item-level table.
order_level_output_path = processed_data_dir / "order_level_clean.csv"  # Define the output path for the cleaned order-level table.

cleaned_item_level.to_csv(item_level_output_path, index=False)  # Save the cleaned item-level table before order-level aggregation.
order_level_clean.to_csv(order_level_output_path, index=False)  # Save the final order-level table for machine learning.

print(f"Saved item-level cleaned table shape: {cleaned_item_level.shape}")  # Print the saved item-level table shape.
print(f"Saved order-level cleaned table shape: {order_level_clean.shape}")  # Print the saved order-level table shape.

Saved item-level cleaned table shape: (110178, 36)
Saved order-level cleaned table shape: (96460, 20)
